<a href="https://colab.research.google.com/github/Anik00223/WasteLens/blob/main/WasteLens_ConvNeXt_XL_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import timm
import os
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
timm: 1.0.30
CUDA: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EXP_DIR = "/content/drive/MyDrive/WasteLens/convnext_xl"

os.makedirs(EXP_DIR, exist_ok=True)
os.makedirs(f"{EXP_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{EXP_DIR}/logs", exist_ok=True)

print("Experiment directory:", EXP_DIR)

Mounted at /content/drive
Experiment directory: /content/drive/MyDrive/WasteLens/convnext_xl


In [ ]:
import torch
import timm
import gc

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda")

model = timm.create_model(
    "hf_hub:timm/convnext_xlarge.fb_in22k_ft_in1k_384",
    pretrained=True,
    num_classes=4
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print("Device:", device)

config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.40GB            

model.safetensors: downloading bytes:           |  0.00B            

AssertionError: Torch not compiled with CUDA enabled

In [1]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("GPU unavailable — don't load the model yet.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA version: 13.0
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [2]:
from pathlib import Path
import subprocess

REPO = Path("/content/WasteLens-ConvNeXt")

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Anik00223/WasteLens.git",
            str(REPO)
        ],
        check=True
    )
else:
    print("Existing repository found:", REPO)

print("\nTop-level files and folders:")
for item in sorted(REPO.iterdir()):
    print(item.name + ("/" if item.is_dir() else ""))

extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
image_counts = {}

for path in REPO.rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:
        folder = str(path.parent.relative_to(REPO))
        image_counts[folder] = image_counts.get(folder, 0) + 1

print("\nImage counts by directory:")
if image_counts:
    for folder, count in sorted(image_counts.items()):
        print(f"{folder}: {count} images")
else:
    print("No image files found inside the repository.")


Top-level files and folders:
.git/
.gitattributes
.gitignore
PROJECT_BRIEF.md
README.md
Untitled2.ipynb
WasteLens_ConvNeXt_XL_Classification.ipynb
data/
docs/
implementation_plan.md
models/
notebooks/
package-lock.json
package.json
requirements.txt
src/
tests/
web/

Image counts by directory:
docs/rejection_experiment: 8 images


In [3]:
from pathlib import Path

REPO = Path("/content/WasteLens-ConvNeXt")

print("=== DATA DIRECTORY ===")
data_dir = REPO / "data"

if data_dir.exists():
    for path in sorted(data_dir.rglob("*")):
        if path.is_file():
            print(f"{path.relative_to(REPO)} ({path.stat().st_size:,} bytes)")
else:
    print("No data/ directory found.")

print("\n=== DATASET / MANIFEST FILES ===")
interesting = {
    ".csv", ".json", ".jsonl", ".parquet",
    ".txt", ".yaml", ".yml", ".zip", ".tar", ".gz"
}

for path in sorted(REPO.rglob("*")):
    if not path.is_file():
        continue
    if any(part in {".git", ".ipynb_checkpoints", "node_modules"}
           for part in path.parts):
        continue
    if path.suffix.lower() in interesting:
        print(path.relative_to(REPO))

print("\n=== GOOGLE DRIVE ===")
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/WasteLens")
print("Mounted:", drive_dir.exists())

if drive_dir.exists():
    for path in sorted(drive_dir.iterdir()):
        print(path.name + ("/" if path.is_dir() else ""))

=== DATA DIRECTORY ===
data/processed/.gitkeep (70 bytes)
data/raw/.gitkeep (55 bytes)

=== DATASET / MANIFEST FILES ===
docs/ood_research.json
docs/realworld_validation.json
docs/rejection_experiment/adapt10_seed_consistency.json
docs/rejection_experiment/adapt10s42_eval.json
docs/rejection_experiment/adapt10s42_gates.json
docs/rejection_experiment/adapt10s42_history.csv
docs/rejection_experiment/adapt10s42_training_config.json
docs/rejection_experiment/adapt10s43_eval.json
docs/rejection_experiment/adapt10s43_gates.json
docs/rejection_experiment/adapt10s43_history.csv
docs/rejection_experiment/adapt10s43_training_config.json
docs/rejection_experiment/adaptation_eval_audit.json
docs/rejection_experiment/adaptation_set_manifest.json
docs/rejection_experiment/dataset_manifest.json
docs/rejection_experiment/eval_sets.json
docs/rejection_experiment/fresh_eval_metrics.json
docs/rejection_experiment/fresh_set_manifest.json
docs/rejection_experiment/frozen_history.csv
docs/rejection_experime

In [4]:
from google.colab import drive
from pathlib import Path
import json

# 1. Mount Google Drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/WasteLens")
REPO = Path("/content/WasteLens-ConvNeXt")

print("\n=== WASTELENS DRIVE FILES ===")

if DRIVE_ROOT.exists():
    for path in sorted(DRIVE_ROOT.iterdir()):
        suffix = "/" if path.is_dir() else ""
        print(path.name + suffix)
else:
    print("No MyDrive/WasteLens folder found.")

# 2. Inspect the dataset manifests
manifest_names = [
    "dataset_manifest.json",
    "adaptation_set_manifest.json",
    "fresh_set_manifest.json",
    "eval_sets.json",
]

print("\n=== MANIFEST STRUCTURE ===")

for name in manifest_names:
    path = REPO / "docs" / "rejection_experiment" / name

    if not path.exists():
        print(f"\n{name}: not found")
        continue

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        print(f"\n{name}")

        if isinstance(data, dict):
            print("Top-level keys:", list(data.keys()))

            for key, value in data.items():
                if isinstance(value, list):
                    print(f"  {key}: {len(value)} entries")
                    if value and isinstance(value[0], dict):
                        print("  First entry fields:", list(value[0].keys()))
                elif isinstance(value, dict):
                    print(f"  {key}: object with keys {list(value.keys())[:12]}")
                elif isinstance(value, (str, int, float, bool)) or value is None:
                    print(f"  {key}: {value}")
        elif isinstance(data, list):
            print("Entries:", len(data))
            if data and isinstance(data[0], dict):
                print("First entry fields:", list(data[0].keys()))

    except Exception as e:
        print("Could not inspect:", str(e))

# 3. Count existing image files in the persistent project folder
print("\n=== DRIVE IMAGE COUNTS ===")

if DRIVE_ROOT.exists():
    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    total = 0

    for folder in DRIVE_ROOT.rglob("*"):
        if folder.is_dir():
            count = sum(
                1 for f in folder.iterdir()
                if f.is_file() and f.suffix.lower() in extensions
            )
            if count:
                print(f"{folder.relative_to(DRIVE_ROOT)}: {count} images")
                total += count

    print("Total image files:", total)

Mounted at /content/drive

=== WASTELENS DRIVE FILES ===
convnext_xl/
iteration12/
models/
training_log.csv

=== MANIFEST STRUCTURE ===

dataset_manifest.json
Top-level keys: ['generated_utc', 'supported_split_seed', 'unsupported_split_seed', 'dedupe', 'sources', 'rejection_head']
  generated_utc: 2026-09-20 13:35
  supported_split_seed: 42
  unsupported_split_seed: 42
  dedupe: object with keys ['method', 'checked_against', 'duplicates_dropped', 'examples']
  sources: 6 entries
  First entry fields: ['source', 'class', 'mapped_label', 'supported', 'split_counts', 'seed', 'note']
  rejection_head: object with keys ['input', 'output', 'bins_head']

adaptation_set_manifest.json
Top-level keys: ['generated_utc', 'protocol', 'source', 'n_images', 'counts_by_group', 'expected_counts', 'rank_offset', 'fresh_titles_skipped', 'corpus_reference_hashes', 'fresh_reference_hashes', 'dedup_matches_corpus', 'fresh', 'entries']
  generated_utc: 2026-09-26 14:52:10 UTC
  protocol: docs/rejection_exper

In [5]:
import json
from pathlib import Path

REPO = Path("/content/WasteLens-ConvNeXt")
DOCS = REPO / "docs" / "rejection_experiment"

# Inspect the six original dataset sources
manifest_path = DOCS / "dataset_manifest.json"
manifest = json.loads(manifest_path.read_text())

print("=== ORIGINAL DATA SOURCES ===")

for i, source in enumerate(manifest["sources"], 1):
    print(f"\nSource {i}")
    print("Name:", source.get("source"))
    print("Original class:", source.get("class"))
    print("Mapped class:", source.get("mapped_label"))
    print("Supported:", source.get("supported"))
    print("Recorded splits:", source.get("split_counts"))
    print("Notes:", source.get("note"))

# Inspect the original training configuration
config_path = DOCS / "training_config.json"

print("\n\n=== ORIGINAL TRAINING CONFIGURATION ===")

if config_path.exists():
    config = json.loads(config_path.read_text())
    print(json.dumps(config, indent=2))
else:
    print("Training configuration not found.")

=== ORIGINAL DATA SOURCES ===

Source 1
Name: kaggle mostafaabla/garbage-classification (12-class)
Original class: 4-bin taxonomy (10 classes mapped, clothes/shoes dropped)
Mapped class: bins 0-3 (supported)
Supported: True
Recorded splits: {'train': 5749, 'val': 1231, 'test': 1233}
Notes: UNCHANGED train.py pipeline; identical membership to the baseline model's data

Source 2
Name: kaggle mostafaabla/garbage-classification (12-class)
Original class: clothes
Mapped class: unsupported
Supported: False
Recorded splits: {'train': 3726, 'val': 798, 'test': 800}
Notes: md5-deduped against ALL supported images + within-source (1 duplicates dropped so far)

Source 3
Name: kaggle mostafaabla/garbage-classification (12-class)
Original class: shoes
Mapped class: unsupported
Supported: False
Recorded splits: {'train': 1383, 'val': 296, 'test': 298}
Notes: md5-deduped against ALL supported images + within-source (1 duplicates dropped so far)

Source 4
Name: synthetic composition of kaggle waste ph

In [6]:
from pathlib import Path
import re

REPO = Path("/content/WasteLens-ConvNeXt")

print("=== TRAINING / DATASET SCRIPTS ===")

for path in sorted(REPO.rglob("*.py")):
    if ".git" in path.parts or "node_modules" in path.parts:
        continue

    if any(word in path.name.lower()
           for word in ("train", "data", "split", "prepare")):
        print(path.relative_to(REPO))

print("\n=== SPLIT LOGIC REFERENCES ===")

pattern = re.compile(
    r"supported_split_seed|train_test_split|random_split|"
    r"StratifiedShuffleSplit|ImageFolder|random_state|split_counts",
    re.IGNORECASE
)

for folder in (REPO / "src", REPO / "tests"):
    if not folder.exists():
        continue

    for path in sorted(folder.rglob("*.py")):
        try:
            lines = path.read_text(encoding="utf-8").splitlines()
        except (OSError, UnicodeError):
            continue

        matches = [
            (i, line.strip())
            for i, line in enumerate(lines, 1)
            if pattern.search(line)
        ]

        if matches:
            print(f"\n--- {path.relative_to(REPO)} ---")
            for line_no, line in matches[:20]:
                print(f"{line_no}: {line}")

=== TRAINING / DATASET SCRIPTS ===
src/train.py
src/train_adaptation.py
src/train_distill_adaptation.py
src/train_head_adaptation.py
src/train_rejection.py
src/train_rejection_frozen.py
src/train_rejection_variantc.py

=== SPLIT LOGIC REFERENCES ===

--- src/train.py ---
36: from sklearn.model_selection import train_test_split
182: train_paths, temp_paths, train_labels, temp_labels = train_test_split(
187: random_state=SEED,
190: val_paths, test_paths, val_labels, test_labels = train_test_split(
195: random_state=SEED,

--- src/train_rejection.py ---
237: "split_counts": {k: len(v) for k, v in splits.items()},
252: "split_counts": {"train": len(tr), "val": len(va),
279: "split_counts": {"train": len(c_tr), "val": len(c_va),
319: "split_counts": {"train": len(nw_tr), "val": len(nw_va),
329: "supported": False, "split_counts": {"train": 0, "val": 0, "test": 10},
409: "supported_split_seed": 42,
410: "unsupported_split_seed": SEED,


In [7]:
from pathlib import Path

REPO = Path("/content/WasteLens-ConvNeXt")

def show_section(path, start, end):
    file = REPO / path
    print(f"\n{'=' * 20} {path}:{start}-{end} {'=' * 20}")

    lines = file.read_text(encoding="utf-8").splitlines()

    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4}: {lines[i - 1]}")

# Dataset configuration and original splitting logic
show_section("src/train.py", 1, 100)
show_section("src/train.py", 140, 215)

# How the original supported and unsupported datasets were assembled
show_section("src/train_rejection.py", 180, 280)
show_section("src/train_rejection.py", 390, 425)


==================== src/train.py:1-100 ====================
   1: # src/train.py
   2: #
   3: # PURPOSE:
   4: #   Train the WasteLens classifier using transfer learning.
   5: #
   6: #   - Loads a pretrained backbone (MobileNetV2, ImageNet weights).
   7: #   - Freezes the backbone and trains a new classification head on the
   8: #     waste categories (recyclable / organic / hazardous / general trash).
   9: #   - Optionally fine-tunes the top layers of the backbone.
  10: #   - Saves checkpoints to models/checkpoints/ and the final Keras model
  11: #     for conversion to TensorFlow.js.
  12: #
  13: # STATUS: Implemented up to the pre-training summary (data load, remap,
  14: #   stratified split, model build, class weights). `model.fit` is NOT
  15: #   called yet - training starts only after explicit user confirmation
  16: #   of the printed split counts.
  17: 
  18: """WasteLens training script.
  19: 
  20: Run `python src/train.py` to:
  21:   1. download the 12-class 

In [9]:
%pip install -q kagglehub

In [10]:
from pathlib import Path
import runpy

REPO = Path("/content/WasteLens-ConvNeXt")

# Load the original WasteLens functions without executing its main workflow.
wl = runpy.run_path(
    str(REPO / "src" / "train.py"),
    run_name="wastelens_data_audit"
)

print("Downloading or retrieving the cached original dataset...")

dataset_root = wl["download_dataset"]()
print("Dataset location:", dataset_root)

print("\nScanning and checking the original image counts...")
images = wl["scan_images"](dataset_root)

print("\nApplying the original four-class mapping...")
mapped_images = wl["remap_to_bins"](images)

print("\nReproducing the original seed-42 split...")
splits = wl["stratified_split"](mapped_images)

wl["report_split_sizes"](splits)

print("\n=== SPLIT SUMMARY ===")
for name in ("train", "val", "test"):
    print(f"{name}: {len(splits[name]):,} images")

total = sum(len(rows) for rows in splits.values())
print(f"Total: {total:,} images")

assert total == 8213, f"Expected 8,213 images, got {total}"
assert [len(splits[k]) for k in ("train", "val", "test")] == [
    5749, 1231, 1233
], "Split counts differ from the original manifest."

print("\n✅ Dataset counts and split totals verified.")

[1/6] Downloading dataset via kagglehub: mostafaabla/garbage-classification
Using Colab cache for faster access to the 'garbage-classification' dataset.
      cached at: /kaggle/input/garbage-classification
Dataset location: /kaggle/input/garbage-classification

Scanning and checking the original image counts...
[2/6] Scanning image files (dropping classes: clothes, shoes)
      per-class scan:
        battery        945  (expected   945)  ok
        biological     985  (expected   985)  ok
        brown-glass    607  (expected   607)  ok
        cardboard      891  (expected   891)  ok
        green-glass    629  (expected   629)  ok
        metal          769  (expected   769)  ok
        paper         1050  (expected  1050)  ok
        plastic        865  (expected   865)  ok
        trash          697  (expected   697)  ok
        white-glass    775  (expected   775)  ok
      scanned total: 8213 (expected 8213)  ok

Applying the original four-class mapping...
[3/6] Remapping 12 cl

In [11]:
from pathlib import Path
from collections import Counter
import csv
import json

REPO = Path("/content/WasteLens-ConvNeXt")
DRIVE_ROOT = Path("/content/drive/MyDrive/WasteLens/convnext_xl")
MANIFEST_DIR = DRIVE_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

dataset_root = Path(dataset_root).resolve()

class_names = wl["BINS"]
expected_counts = {"train": 5749, "val": 1231, "test": 1233}

records = []
split_paths = {}

for split_name, rows in splits.items():
    split_paths[split_name] = set()

    for image_path, label_idx in rows:
        image_path = Path(image_path).resolve()
        relative_path = image_path.relative_to(dataset_root).as_posix()

        split_paths[split_name].add(relative_path)

        records.append({
            "split": split_name,
            "relative_path": relative_path,
            "label_idx": int(label_idx),
            "label": class_names[int(label_idx)],
        })

for split_name, expected in expected_counts.items():
    actual = sum(r["split"] == split_name for r in records)
    assert actual == expected, (split_name, actual, expected)

assert not (split_paths["train"] & split_paths["val"])
assert not (split_paths["train"] & split_paths["test"])
assert not (split_paths["val"] & split_paths["test"])

csv_path = MANIFEST_DIR / "supported_split_seed42.csv"

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["split", "relative_path", "label_idx", "label"]
    )
    writer.writeheader()
    writer.writerows(records)

summary = {
    "experiment": "WasteLens ConvNeXt-XL",
    "dataset": "mostafaabla/garbage-classification",
    "split_seed": 42,
    "class_order": class_names,
    "split_counts": expected_counts,
    "total_images": len(records),
    "split_overlap": False,
    "manifest_file": csv_path.name,
}

summary_path = MANIFEST_DIR / "supported_split_seed42_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("✅ Manifest saved to Google Drive")
print(csv_path)
print(summary_path)
print("Total images:", len(records))
print("Train/val/test overlap: none")

✅ Manifest saved to Google Drive
/content/drive/MyDrive/WasteLens/convnext_xl/manifests/supported_split_seed42.csv
/content/drive/MyDrive/WasteLens/convnext_xl/manifests/supported_split_seed42_summary.json
Total images: 8213
Train/val/test overlap: none


In [12]:
import gc
import torch
import timm

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from collections import Counter

# Remove earlier stress-test model/optimizer if they still exist.
if "optimizer" in globals():
    del optimizer
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

assert torch.cuda.is_available(), "GPU is not available."

device = torch.device("cuda")

# Fresh pretrained backbone and freshly initialized four-class head.
model = timm.create_model(
    "hf_hub:timm/convnext_xlarge.fb_in22k_ft_in1k_384",
    pretrained=True,
    num_classes=4,
).to(device)

data_cfg = timm.data.resolve_model_data_config(model)
image_size = data_cfg["input_size"][-1]
mean, std = data_cfg["mean"], data_cfg["std"]

print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Input resolution:", image_size)
print("Normalization mean:", mean)
print("Normalization std:", std)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        image_size,
        scale=(0.8, 1.0),
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (image_size, image_size),
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class WasteDataset(Dataset):
    def __init__(self, rows, root, transform):
        self.rows = rows
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        path = self.root / row["relative_path"]

        with Image.open(path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["label_idx"])

train_rows = [r for r in records if r["split"] == "train"]
val_rows = [r for r in records if r["split"] == "val"]
test_rows = [r for r in records if r["split"] == "test"]

BATCH_SIZE = 8

train_loader = DataLoader(
    WasteDataset(train_rows, dataset_root, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    WasteDataset(val_rows, dataset_root, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    WasteDataset(test_rows, dataset_root, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

images, labels = next(iter(train_loader))

print("\n✅ DataLoader smoke test passed")
print("Images:", tuple(images.shape))
print("Labels:", tuple(labels.shape))
print("Labels in first batch:", labels.tolist())
print("Image tensor dtype:", images.dtype)
print("Train batches per epoch:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Model parameters: 348,156,164
Input resolution: 384
Normalization mean: [0.485, 0.456, 0.406]
Normalization std: [0.229, 0.224, 0.225]

✅ DataLoader smoke test passed
Images: (8, 3, 384, 384)
Labels: (8,)
Labels in first batch: [0, 3, 0, 0, 0, 3, 0, 0]
Image tensor dtype: torch.float32
Train batches per epoch: 719
Validation batches: 154
Test batches: 155


In [13]:
import torch
import gc

assert torch.cuda.is_available(), "CUDA is unavailable."

model.train()
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.05
)

scaler = torch.amp.GradScaler("cuda")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

images, labels = next(iter(train_loader))
images = images.to("cuda", non_blocking=True)
labels = labels.to("cuda", non_blocking=True)

try:
    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        outputs = model(images)
        loss = criterion(outputs, labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print("✅ Real-data training smoke test passed")
    print("Output shape:", tuple(outputs.shape))
    print("Loss:", loss.item())
    print(
        "Peak VRAM:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

except torch.cuda.OutOfMemoryError:
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    print("❌ Out of memory — do not start full training.")
    raise

finally:
    del images, labels

✅ Real-data training smoke test passed
Output shape: (8, 4)
Loss: 1.391357421875
Peak VRAM: 7.67 GB


In [14]:
import os
import gc
import csv
import json
import shutil
from pathlib import Path

import torch
import timm
import numpy as np

from sklearn.metrics import accuracy_score, f1_score

# ---------------- CONFIGURATION ----------------

EXP_DIR = Path("/content/drive/MyDrive/WasteLens/convnext_xl")
CHECKPOINT_DIR = EXP_DIR / "checkpoints"
LOG_DIR = EXP_DIR / "logs"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

LAST_CKPT = CHECKPOINT_DIR / "convnext_xl_last.pt"
BEST_WEIGHTS = CHECKPOINT_DIR / "convnext_xl_best.pt"

HISTORY_JSON = LOG_DIR / "training_history.json"
HISTORY_CSV = LOG_DIR / "training_history.csv"

MODEL_NAME = "hf_hub:timm/convnext_xlarge.fb_in22k_ft_in1k_384"
CLASS_NAMES = ["recyclable", "organic", "hazardous", "general trash"]

NUM_CLASSES = 4
NUM_EPOCHS = 10
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.05

device = torch.device("cuda")
assert torch.cuda.is_available(), "CUDA is unavailable."
assert len(train_loader.dataset) == 5749
assert len(val_loader.dataset) == 1231
assert len(test_loader.dataset) == 1233

# ---------------- FRESH MODEL OR RESUME ----------------

gc.collect()
torch.cuda.empty_cache()

resume = LAST_CKPT.exists()

model = timm.create_model(
    MODEL_NAME,
    pretrained=not resume,
    num_classes=NUM_CLASSES,
).to(device)

classifier = model.get_classifier()
head_ids = {id(p) for p in classifier.parameters()}

backbone_params = [
    p for p in model.parameters()
    if p.requires_grad and id(p) not in head_ids
]
head_params = list(classifier.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS
)
scaler = torch.amp.GradScaler("cuda")

# Inverse-frequency weights from the original WasteLens class frequencies.
class_weights = torch.tensor(
    [0.36758312, 2.08599419, 2.17107251, 2.94518443],
    dtype=torch.float32,
    device=device,
)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

history = []
best_macro_f1 = -1.0
global_step = 0
start_epoch = 0

if resume:
    print("Resuming from the latest Drive checkpoint...")
    checkpoint = torch.load(
        LAST_CKPT,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    history = checkpoint["history"]
    best_macro_f1 = checkpoint["best_macro_f1"]
    global_step = checkpoint["global_step"]
    start_epoch = checkpoint["epoch_completed"]

    del checkpoint
    gc.collect()
    torch.cuda.empty_cache()

else:
    print("Starting from fresh pretrained ConvNeXt-XL weights.")
    print("The earlier smoke-test weights will NOT be reused.")

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Starting epoch: {start_epoch + 1}")
print(f"Target epochs: {NUM_EPOCHS}")

# ---------------- SAFE DRIVE CHECKPOINT WRITER ----------------

def atomic_save_to_drive(obj, destination):
    """Write locally, copy to Drive, then replace the destination."""
    destination = Path(destination)
    local_temp = Path("/content") / f"{destination.stem}_staging{destination.suffix}"
    drive_temp = destination.with_name(destination.name + ".tmp")

    try:
        torch.save(obj, local_temp)
        shutil.copyfile(local_temp, drive_temp)
        os.replace(drive_temp, destination)
    finally:
        if local_temp.exists():
            local_temp.unlink()
        if drive_temp.exists():
            drive_temp.unlink()

def save_history():
    HISTORY_JSON.write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )

    if history:
        with HISTORY_CSV.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
            writer.writeheader()
            writer.writerows(history)

# ---------------- TRAIN / VALIDATE ----------------

for epoch_idx in range(start_epoch, NUM_EPOCHS):
    epoch_num = epoch_idx + 1
    model.train()

    running_loss = 0.0
    train_targets = []
    train_preds = []

    torch.cuda.reset_peak_memory_stats()

    print(f"\n===== EPOCH {epoch_num}/{NUM_EPOCHS} =====")

    for batch_idx, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch_num}, "
                f"batch {batch_idx}."
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        train_preds.extend(logits.detach().argmax(1).cpu().tolist())
        train_targets.extend(labels.detach().cpu().tolist())

        global_step += 1

        if batch_idx % 50 == 0 or batch_idx == len(train_loader):
            print(
                f"Epoch {epoch_num} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss {loss.item():.4f} | "
                f"Peak VRAM "
                f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
            )

        del images, labels, logits, loss

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = accuracy_score(train_targets, train_preds)

    # Validation: never use the test loader for model selection.
    model.eval()
    val_loss_sum = 0.0
    val_targets = []
    val_preds = []

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)

            val_loss_sum += loss.item() * images.size(0)
            val_preds.extend(logits.argmax(1).cpu().tolist())
            val_targets.extend(labels.cpu().tolist())

    val_loss = val_loss_sum / len(val_loader.dataset)
    val_acc = accuracy_score(val_targets, val_preds)
    val_macro_f1 = f1_score(
        val_targets,
        val_preds,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0,
    )

    row = {
        "epoch": epoch_num,
        "train_loss": train_loss,
        "train_accuracy": float(train_acc),
        "val_loss": val_loss,
        "val_accuracy": float(val_acc),
        "val_macro_f1": float(val_macro_f1),
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"],
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
    }

    history.append(row)
    improved = val_macro_f1 > best_macro_f1

    if improved:
        best_macro_f1 = float(val_macro_f1)
        atomic_save_to_drive(model.state_dict(), BEST_WEIGHTS)
        print(f"✅ New best validation macro-F1: {best_macro_f1:.4f}")

    scheduler.step()

    # Full checkpoint permits continuation from the last completed epoch.
    checkpoint = {
        "epoch_completed": epoch_num,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "best_macro_f1": best_macro_f1,
        "model_name": MODEL_NAME,
        "class_names": CLASS_NAMES,
        "input_size": 384,
        "split_manifest": str(
            EXP_DIR / "manifests" / "supported_split_seed42.csv"
        ),
    }

    atomic_save_to_drive(checkpoint, LAST_CKPT)
    save_history()

    print(
        f"Epoch {epoch_num} summary | "
        f"train loss={train_loss:.4f} | "
        f"train accuracy={train_acc:.4f} | "
        f"val loss={val_loss:.4f} | "
        f"val accuracy={val_acc:.4f} | "
        f"val macro-F1={val_macro_f1:.4f}"
    )
    print("✅ Drive checkpoint and training history saved.")

    del checkpoint
    gc.collect()
    torch.cuda.empty_cache()

print("\n🎉 Training loop completed.")
print("Best validation macro-F1:", best_macro_f1)
print("Best weights:", BEST_WEIGHTS)
print("Latest resumable checkpoint:", LAST_CKPT)
print("History CSV:", HISTORY_CSV)

Starting from fresh pretrained ConvNeXt-XL weights.
The earlier smoke-test weights will NOT be reused.
Total parameters: 348,156,164
Starting epoch: 1
Target epochs: 10

===== EPOCH 1/10 =====
Epoch 1 | Batch 50/719 | Loss 0.4465 | Peak VRAM 12.86 GB
Epoch 1 | Batch 100/719 | Loss 0.1131 | Peak VRAM 12.86 GB
Epoch 1 | Batch 150/719 | Loss 0.0917 | Peak VRAM 12.86 GB
Epoch 1 | Batch 200/719 | Loss 0.0616 | Peak VRAM 12.86 GB
Epoch 1 | Batch 250/719 | Loss 0.0383 | Peak VRAM 12.86 GB
Epoch 1 | Batch 300/719 | Loss 0.0403 | Peak VRAM 12.86 GB
Epoch 1 | Batch 350/719 | Loss 0.1021 | Peak VRAM 12.86 GB
Epoch 1 | Batch 400/719 | Loss 0.0069 | Peak VRAM 12.86 GB
Epoch 1 | Batch 450/719 | Loss 0.0064 | Peak VRAM 12.86 GB
Epoch 1 | Batch 500/719 | Loss 0.0146 | Peak VRAM 12.86 GB
Epoch 1 | Batch 550/719 | Loss 0.0051 | Peak VRAM 12.86 GB
Epoch 1 | Batch 600/719 | Loss 0.0057 | Peak VRAM 12.86 GB
Epoch 1 | Batch 650/719 | Loss 0.0026 | Peak VRAM 12.86 GB
Epoch 1 | Batch 700/719 | Loss 0.0077 | P

In [15]:
from pathlib import Path
import json

BASE = Path("/content/drive/MyDrive/WasteLens/convnext_xl")

required_files = {
    "Best model weights":
        BASE / "checkpoints/convnext_xl_best.pt",
    "Latest resumable checkpoint":
        BASE / "checkpoints/convnext_xl_last.pt",
    "Training history JSON":
        BASE / "logs/training_history.json",
    "Training history CSV":
        BASE / "logs/training_history.csv",
    "Dataset split manifest":
        BASE / "manifests/supported_split_seed42.csv",
    "Dataset summary":
        BASE / "manifests/supported_split_seed42_summary.json",
}

print("========= WASTELENS BACKUP CHECK =========")

all_ok = True

for name, path in required_files.items():
    exists = path.is_file() and path.stat().st_size > 0
    status = "PASS ✅" if exists else "MISSING ❌"

    print(f"{status} | {name}")

    if exists:
        print(f"       Size: {path.stat().st_size / 1024**2:.2f} MB")
    else:
        all_ok = False

history_path = BASE / "logs/training_history.json"

if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))

    print("\n========= TRAINING SUMMARY =========")
    print("Completed epochs:", len(history))

    if history:
        best = max(history, key=lambda r: r["val_macro_f1"])
        print("Best epoch:", best["epoch"])
        print(f"Best validation accuracy: {best['val_accuracy']:.4%}")
        print(f"Best validation macro-F1: {best['val_macro_f1']:.4f}")

print("\nBackup verification:", "ALL PASS ✅" if all_ok else "INCOMPLETE ❌")

if not all_ok:
    print("Do not shut down until we investigate missing files.")

========= WASTELENS BACKUP CHECK =========
PASS ✅ | Best model weights
       Size: 1328.24 MB
PASS ✅ | Latest resumable checkpoint
       Size: 3984.76 MB
PASS ✅ | Training history JSON
       Size: 0.00 MB
PASS ✅ | Training history CSV
       Size: 0.00 MB
PASS ✅ | Dataset split manifest
       Size: 0.53 MB
PASS ✅ | Dataset summary
       Size: 0.00 MB

========= TRAINING SUMMARY =========
Completed epochs: 10
Best epoch: 2
Best validation accuracy: 99.7563%
Best validation macro-F1: 0.9967

Backup verification: ALL PASS ✅


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -q \
    torch==2.11.0 \
    torchvision==0.26.0 \
    --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 24.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 63.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 70.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 54.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: False


In [ ]:
import torch
import timm

device = torch.device("cuda")

model = timm.create_model(
    "hf_hub:timm/convnext_xlarge.fb_in22k_ft_in1k_384",
    pretrained=True,
    num_classes=4
).to(device)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
!nvidia-smi

Tue Oct  6 18:49:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
CUDA devices: 1
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
import torch
import timm

device = torch.device("cuda")

model = timm.create_model(
    "hf_hub:timm/convnext_xlarge.fb_in22k_ft_in1k_384",
    pretrained=True,
    num_classes=4
)

model = model.to(device)

print("Parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Parameters: 348,156,164


In [ ]:
import os

REPO = "/content/WasteLens-ConvNeXt"

for root, dirs, files in os.walk(REPO):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]

    if image_files:
        print(f"{root} -> {len(image_files)} images")